SETUP

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DateType
from config.f1_config import add_ingestion_metadata,CATALOG

API_URL = "https://api.jolpi.ca/ergast/f1/current/circuits.json"
STAGING_PATH = "abfss://staging@stf1project.dfs.core.windows.net/circuits"
BRONZE_TABLE = "f1_workspace.bronze.circuits"

season = datetime.now().year
run_date = datetime.now().strftime("%Y-%m-%d")

API CALL

In [0]:
response = requests.get(API_URL, timeout=30)
response.raise_for_status()

data = response.json()
circuits_list = data["MRData"]["CircuitTable"]["Circuits"]

print(f"Pulled {len(circuits_list)} circuits for season {season}")

LAND DATA IN ADLS


In [0]:
raw_path = f"{STAGING_PATH}/season={season}/run_date={run_date}/circuits.json"

dbutils.fs.put(raw_path, json.dumps(circuits_list), overwrite=True)

print(f"Landed raw JSON at: {raw_path}")

EXPLICIT SCHEMA


In [0]:
circuit_schema = StructType([
    StructField("circuitId", StringType(), True),
    StructField("circuitName", StringType(), True),
    StructField("url", StringType(), True),
    StructField("Location", StructType([
        StructField("lat", StringType(), True),
        StructField("long", StringType(), True),
        StructField("locality", StringType(), True),
        StructField("country", StringType(), True),
    ]), True),
])



READ DATA

In [0]:
df = spark.read.schema(circuit_schema).json(raw_path)
df = add_ingestion_metadata(df)


WRITE DATA

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(BRONZE_TABLE)
